#Ingerir o arquivo Sprints.json
1- Ler o arquivo usando a API de leitura de DataFrame do Spark

2-  Definir e aplicar o Schema 

3- Adicionar colunas de metadados
- Arquivo de origem
- Timestamp (data/hora) de ingestão

4- Escrever/salvar na tabela Delta da camada bronze

Nota: O JSON está em formato de múltiplas linhas.

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name =f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType 

sprints_schema = StructType ([
    StructField ('date', StringType()),
    StructField ('raceName', StringType()),
    StructField ('round', IntegerType()),
    StructField ('season', IntegerType()),
    StructField ('url', StringType()),
    StructField ('constructorId', StringType()),
    StructField ('driverId', StringType()),
    StructField ('grid', IntegerType()),
    StructField ('laps', IntegerType()),
    StructField ('number', IntegerType()),
    StructField ('points', FloatType()),
    StructField ('position', IntegerType()),
    StructField ('positiontext', StringType()),
    StructField ('status', StringType())
])

In [0]:
sprints_df = (
    spark.read
    .format('json')
    .schema(sprints_schema) 
    .option('mode', 'FAILFAST')
    .option('multiLine', True)
    .load(source_file) 
)

In [0]:
display(sprints_df)

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positiontext,status
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,perez,2,17,11,8.0,1,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,leclerc,1,17,16,7.0,2,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,max_verstappen,3,17,1,6.0,3,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,russell,4,17,63,5.0,4,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,sainz,5,17,55,4.0,5,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,alonso,8,17,14,3.0,6,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,hamilton,6,17,44,2.0,7,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,stroll,9,17,18,1.0,8,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,williams,albon,7,17,23,0.0,9,null,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mclaren,piastri,11,17,81,0.0,10,null,Finished


In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
(
    sprints_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.read.table(table_name))

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positiontext,status,ingestion_timestamp,source_file
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,red_bull,max_verstappen,2,17,33,3.0,1,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,mercedes,hamilton,1,17,44,2.0,2,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,mercedes,bottas,3,17,77,1.0,3,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,ferrari,leclerc,4,17,16,0.0,4,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,mclaren,norris,6,17,4,0.0,5,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,mclaren,ricciardo,7,17,3,0.0,6,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,alpine,alonso,11,17,14,0.0,7,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,aston_martin,vettel,10,17,5,0.0,8,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,williams,russell,8,17,63,0.0,9,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,https://en.wikipedia.org/wiki/2021_British_Grand_Prix,alpine,ocon,13,17,31,0.0,10,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json


In [0]:
%sql
SELECT season, COUNT(*)
FROM formula1.bronze.sprints
GROUP BY season
ORDER BY season;

season,COUNT(*)
null,7
2021,73
2022,60
2023,120
2024,120
2025,120
